# Notebook 08 — Held-Out Test Evaluation and Promotion Gate

**AI Interview Assistant · Machine Learning Pipeline, Stage 8 of 9**

---

## Purpose

Open the sealed test split — **once** — and decide whether the model is good
enough to be promoted to production.

## This is the only notebook authorised to read the test split

Stage 4 sealed `test.jsonl` with a SHA-256 hash and an allow-list containing
exactly one notebook id: **8**. Step 1 calls the guard, which verifies both that
this notebook is authorised and that the file has not been altered since it was
sealed.

Why this discipline matters: a test score consulted repeatedly during
development, with the best run reported, is not a test score. It is a training
score wearing a disguise. The seal makes that mistake structurally impossible
rather than a matter of self-restraint.

## What is measured

| Metric | What it captures | Why it is included |
|---|---|---|
| Cross-entropy loss | per-token prediction error | the training objective, on unseen data |
| Perplexity | effective branching factor | interpretable form of the loss |
| Top-1 / Top-5 accuracy | next-token prediction hit rate | is the model's *best guess* right? |
| Per-domain loss | where the model is weak | a mean hides an unusable subgroup |
| Per-difficulty loss | does difficulty affect prediction? | tests the conditioning |
| Generation quality | is the output usable at all? | loss can improve while output stays unusable |

Reporting only aggregate loss would hide a model that works on the two largest
domains and fails on the rest, so the per-group breakdown is not optional.

## Promotion gate

The model is promoted only if it passes **every** declared criterion. The
thresholds are set before the test split is read.

## Outputs

- `reports/fine_tuned_model_evaluation.json`
- `reports/figures/08_*.png`

---

In [ ]:
NOTEBOOK_ID = 8

# ─────────────────────────────────────────────────────────────────────────────
# Step 0 — Environment bootstrap
#
# Locates the project workspace so this notebook runs unchanged in Google Colab,
# a local Jupyter server, or VS Code. Every later step resolves its paths from
# WORKSPACE_DIR, so nothing below depends on where the notebook was opened.
# ─────────────────────────────────────────────────────────────────────────────
import os
import sys
import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

def locate_workspace() -> Path:
    """Return the ml-service directory, whatever environment we are in."""
    # 1. Google Colab: mount Drive so checkpoints survive a runtime restart.
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        ws = Path("/content/drive/MyDrive/ai-interview-system/ml-service")
        ws.mkdir(parents=True, exist_ok=True)
        print("Environment      : Google Colab (Drive mounted)")
        return ws
    except ImportError:
        pass

    # 2. Local: walk up from the notebook until we find the ml-service root,
    #    identified by the dataset directory it must contain.
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "dataset").is_dir() and (candidate / "notebooks").is_dir():
            print("Environment      : local")
            return candidate
    print("Environment      : local (fallback to cwd)")
    return here

WORKSPACE_DIR = locate_workspace()
os.chdir(WORKSPACE_DIR)
if str(WORKSPACE_DIR) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_DIR))

# Canonical paths used across all nine notebooks.
RAW_DIR       = WORKSPACE_DIR / "dataset" / "raw"
PROCESSED_DIR = WORKSPACE_DIR / "dataset" / "processed"
QG_DIR        = PROCESSED_DIR / "question_generator"
SPLIT_DIR     = PROCESSED_DIR / "splits"
TOKENIZER_DIR = WORKSPACE_DIR / "tokenizer"
CKPT_DIR      = WORKSPACE_DIR / "checkpoints"
MODEL_DIR     = WORKSPACE_DIR / "models"
REPORTS_DIR   = WORKSPACE_DIR / "reports"
FIGURES_DIR   = REPORTS_DIR / "figures"

for d in (RAW_DIR, PROCESSED_DIR, SPLIT_DIR, TOKENIZER_DIR, CKPT_DIR,
          MODEL_DIR, REPORTS_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Workspace        : {WORKSPACE_DIR}")
print(f"Python           : {sys.version.split()[0]}")
print(f"Run started      : {datetime.now(timezone.utc).isoformat(timespec='seconds')}")

---

## Step 0b — Figure and statistics conventions

One style definition serves every figure in the nine-notebook pipeline, so
charts are directly comparable when placed side by side in the write-up.

Three conventions are fixed here:

1. **A colour-blind-safe categorical palette** — the same six colours, in the
   same order, wherever a chart encodes categories.
2. **Automatic figure export** — `save_figure()` writes every figure to
   `reports/figures/` at 200 dpi with a numbered filename, and prints its
   caption, so figures can be cited as *Figure N.k* in the dissertation.
3. **A single summary-statistics function** — `describe_series()` reports
   n, mean, sd, the five-number summary, skewness and kurtosis in a fixed
   order for every variable, so distributions are described consistently.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 0b — Plotting conventions
#
# One style definition for every figure in the pipeline, so figures across the
# nine notebooks are directly comparable in the dissertation. Every figure is
# also saved to reports/figures/ at 200 dpi, ready to drop into the write-up.
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.edgecolor": "#444444",
    "grid.alpha": 0.3,
    "legend.frameon": True,
    "figure.autolayout": False,
})

# Colour-blind-safe categorical palette, used consistently for every chart.
PALETTE = ["#3B6FD4", "#E1893B", "#3EA37A", "#C4576B", "#7B5EA7", "#8C7B68"]
sns.set_palette(PALETTE)

_figure_index = {"n": 0}

def save_figure(fig, slug: str, caption: str = "") -> Path:
    """Save a figure with a numbered filename and print its caption."""
    _figure_index["n"] += 1
    n = _figure_index["n"]
    path = FIGURES_DIR / f"{NOTEBOOK_ID:02d}_fig{n:02d}_{slug}.png"
    fig.savefig(path)
    label = f"Figure {NOTEBOOK_ID}.{n}"
    if caption:
        print(f"{label}: {caption}")
    print(f"           saved -> {path.relative_to(WORKSPACE_DIR)}")
    return path

def describe_series(series: pd.Series, name: str) -> pd.Series:
    """Summary statistics reported in a consistent order for every variable."""
    s = pd.to_numeric(series, errors="coerce").dropna()
    return pd.Series({
        "n": len(s),
        "mean": s.mean(),
        "std": s.std(ddof=1),
        "min": s.min(),
        "q1": s.quantile(0.25),
        "median": s.median(),
        "q3": s.quantile(0.75),
        "max": s.max(),
        "skew": s.skew(),
        "kurtosis": s.kurtosis(),
    }, name=name)

print("Plot style       : configured")
print(f"Figure output    : {FIGURES_DIR.relative_to(WORKSPACE_DIR)}")
print(f"Palette          : {len(PALETTE)} colour-blind-safe categories")

---

## Step 1 — Authorised access to the sealed test split

The guard checks two things: that this notebook is on the allow-list, and that
the test file's SHA-256 still matches the seal recorded in Stage 4. A mismatch
means the file changed after sealing, which invalidates the whole experiment.

In [ ]:
import math
import time
import hashlib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformer_scratch import CustomBPETokenizer, load_checkpoint

LOCK_FILE = SPLIT_DIR / "test_lock.json"
assert LOCK_FILE.exists(), (
    f"{LOCK_FILE.name} missing — run Notebook 04 to seal the test split."
)

def guard_test_access(notebook_id: int) -> dict:
    """Verify authorisation and seal integrity before the test split is read."""
    seal = json.loads(LOCK_FILE.read_text(encoding="utf-8"))

    if notebook_id not in seal["authorised_notebook_ids"]:
        raise PermissionError(
            f"Notebook {notebook_id} is not authorised to read the test split. "
            f"Authorised: {seal['authorised_notebook_ids']}."
        )

    test_path = SPLIT_DIR / seal["file"]
    current_hash = hashlib.sha256(test_path.read_bytes()).hexdigest()
    if current_hash != seal["sha256"]:
        raise RuntimeError(
            "TEST SPLIT MODIFIED SINCE SEALING.\n"
            f"  sealed  : {seal['sha256']}\n"
            f"  current : {current_hash}\n"
            "The experiment is no longer valid. Re-run Notebooks 03-07 to "
            "rebuild the splits."
        )
    return seal

print("TEST SPLIT ACCESS CONTROL")
print("=" * 78)
seal = guard_test_access(NOTEBOOK_ID)
print(f"  Requesting notebook : {NOTEBOOK_ID}")
print(f"  Authorised          : {seal['authorised_notebook_ids']}  -> GRANTED")
print(f"  Sealed at           : {seal['sealed_utc']}")
print(f"  Seal SHA-256        : {seal['sha256'][:48]}...")
print(f"  Integrity           : VERIFIED (file unchanged since sealing)")
print(f"  Records             : {seal['records']:,}")
print("=" * 78)
print("\nFIRST AND ONLY AUTHORISED READ OF THE TEST SPLIT.")

test_records = [json.loads(line) for line in
                (SPLIT_DIR / seal["file"]).read_text(encoding="utf-8")
                .splitlines() if line.strip()]
val_records = [json.loads(line) for line in
               (SPLIT_DIR / "validation.jsonl").read_text(encoding="utf-8")
               .splitlines() if line.strip()]

print(f"\nTest records       : {len(test_records):,}")
print(f"Validation records : {len(val_records):,}  (for the overfit check)")

---

## Step 2 — Load the model Stage 7 nominated

Stage 7 decides whether the specialised or the base model goes forward, based on
whether specialisation actually improved validation loss. This notebook honours
that decision rather than re-choosing — re-choosing here, with test data
available, would be exactly the circularity the seal exists to prevent.

In [ ]:
SPEC_REPORT = REPORTS_DIR / "specialization_report.json"
SELECTION_FILE = REPORTS_DIR / "model_selection.json"
assert SELECTION_FILE.exists(), "Run Notebook 06 first."

selection = json.loads(SELECTION_FILE.read_text(encoding="utf-8"))

if SPEC_REPORT.exists():
    spec_report = json.loads(SPEC_REPORT.read_text(encoding="utf-8"))
    verdict = spec_report["verdict"]
    MODEL_KIND = verdict["model_for_stage_8"]
    CKPT_PATH = WORKSPACE_DIR / verdict["checkpoint_for_stage_8"]
    print(f"Stage 7 nominated the {MODEL_KIND.upper()} model.")
    print(f"  Reason: specialisation "
          f"{'improved' if verdict['specialisation_helped'] else 'did not improve'}"
          f" validation loss.")
else:
    MODEL_KIND = "base"
    CKPT_PATH = WORKSPACE_DIR / selection["selected"]["checkpoint"]
    print("No Stage 7 report found — evaluating the Stage 6 base model.")

assert CKPT_PATH.exists(), f"Checkpoint {CKPT_PATH} not found."

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

tokenizer = CustomBPETokenizer.load(TOKENIZER_DIR)
model, payload = load_checkpoint(CKPT_PATH, device=DEVICE)
model.eval()

print(f"\nMODEL UNDER TEST")
print("=" * 74)
print(f"  Kind          : {MODEL_KIND}")
print(f"  Architecture  : {selection['selected']['label']}")
print(f"  Checkpoint    : {CKPT_PATH.relative_to(WORKSPACE_DIR)}")
print(f"  Parameters    : {model.count_parameters():,}")
print(f"  Trained epochs: {payload.get('epoch', 'unknown')}")
print(f"  Device        : {DEVICE}")
print("=" * 74)

---

## Step 3 — Build the test tensors

The test data is formatted **identically** to the training data. Any difference
in formatting would make the comparison invalid — the model would be penalised
for a mismatch rather than for a genuine lack of skill.

In [ ]:
CONDITIONED = (MODEL_KIND == "specialised")

def format_record(record: dict) -> str:
    if CONDITIONED:
        return (f"<DOMAIN: {record['domain']}> "
                f"<DIFFICULTY: {record['difficulty']}> {record['question']}")
    return record["question"]

MAX_SEQ_LEN = int(json.loads(
    (REPORTS_DIR / "candidate_training_report.json").read_text(encoding="utf-8")
)["data"]["max_seq_len"])
if CONDITIONED:
    MAX_SEQ_LEN = min(512, MAX_SEQ_LEN + 24)   # room for the control tokens

PAD_ID = 0

class EvalDataset(Dataset):
    """Test sequences, retaining each record's labels for the group breakdown."""

    def __init__(self, records, tokenizer, max_len):
        self.samples, self.meta = [], []
        for record in records:
            ids = tokenizer.encode(format_record(record),
                                   add_special_tokens=True)[:max_len]
            if len(ids) < 2:
                continue
            self.samples.append(ids)
            self.meta.append({
                "domain": record["domain"],
                "difficulty": record["difficulty"],
                "question": record["question"],
            })
        self.max_len = max_len

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        ids = self.samples[index]
        padded = ids + [PAD_ID] * (self.max_len - len(ids))
        tensor = torch.tensor(padded, dtype=torch.long)
        targets = tensor.clone()
        targets[len(ids):] = -100
        return tensor[:-1], targets[1:], index

test_dataset = EvalDataset(test_records, tokenizer, MAX_SEQ_LEN)
val_dataset = EvalDataset(val_records, tokenizer, MAX_SEQ_LEN)

BATCH_SIZE = 16 if DEVICE == "cuda" else 8
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("TEST TENSORS")
print("=" * 66)
print(f"  Format          : "
      f"{'conditioned (control tokens)' if CONDITIONED else 'plain question'}")
print(f"  Test sequences  : {len(test_dataset):,}")
print(f"  Val sequences   : {len(val_dataset):,}")
print(f"  Context window  : {MAX_SEQ_LEN}")
print(f"  Batch size      : {BATCH_SIZE}")
print("=" * 66)
print("\nFormatting matches Stage 5/7 training exactly — the model is not "
      "penalised\nfor a format it never saw.")

---

## Step 4 — Evaluate: loss, perplexity, and top-k accuracy

Top-k accuracy is reported alongside loss because they answer different
questions. Loss measures how much probability mass the model puts on the right
token; **top-1 accuracy** measures how often the right token is its actual best
guess — which is what matters for generation.

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=-100, reduction="sum")

def evaluate_detailed(model, loader, dataset) -> dict:
    """Aggregate metrics plus a per-record loss, needed for the group breakdown."""
    model.eval()
    total_loss, total_tokens = 0.0, 0
    top1_hits, top5_hits = 0, 0
    per_record = {}
    start = time.perf_counter()

    with torch.no_grad():
        for inputs, targets, indices in loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            logits = model(inputs)
            if isinstance(logits, tuple):
                logits = logits[0]

            flat_logits = logits.reshape(-1, logits.size(-1))
            flat_targets = targets.reshape(-1)
            valid = flat_targets != -100

            total_loss += float(criterion(flat_logits, flat_targets))
            total_tokens += int(valid.sum())

            # Top-k over scored positions only.
            top5 = flat_logits[valid].topk(5, dim=-1).indices
            gold = flat_targets[valid].unsqueeze(-1)
            top1_hits += int((top5[:, :1] == gold).any(dim=-1).sum())
            top5_hits += int((top5 == gold).any(dim=-1).sum())

            # Per-sequence mean loss, for the domain/difficulty breakdown.
            token_losses = nn.functional.cross_entropy(
                flat_logits, flat_targets, ignore_index=-100, reduction="none"
            ).reshape(targets.shape)
            mask = (targets != -100).float()
            seq_loss = ((token_losses * mask).sum(dim=1)
                        / mask.sum(dim=1).clamp(min=1))
            for position, record_index in enumerate(indices.tolist()):
                per_record[record_index] = float(seq_loss[position])

    seconds = time.perf_counter() - start
    mean_loss = total_loss / max(total_tokens, 1)
    return {
        "loss": mean_loss,
        "perplexity": float(math.exp(min(mean_loss, 20))),
        "top1_accuracy": top1_hits / max(total_tokens, 1) * 100,
        "top5_accuracy": top5_hits / max(total_tokens, 1) * 100,
        "tokens_scored": total_tokens,
        "sequences": len(dataset),
        "seconds": seconds,
        "tokens_per_second": total_tokens / max(seconds, 1e-9),
        "per_record_loss": per_record,
    }

print("Evaluating on the validation split (reference)...")
val_metrics = evaluate_detailed(model, val_loader, val_dataset)
print("Evaluating on the HELD-OUT TEST split...")
test_metrics = evaluate_detailed(model, test_loader, test_dataset)

print("\nRESULTS")
print("=" * 84)
print(f"{'Metric':<26} {'Validation':>14} {'Test':>14} {'Difference':>14}")
print("-" * 84)
for key, label, fmt in [
    ("loss", "Cross-entropy loss", "{:.4f}"),
    ("perplexity", "Perplexity", "{:.2f}"),
    ("top1_accuracy", "Top-1 accuracy (%)", "{:.2f}"),
    ("top5_accuracy", "Top-5 accuracy (%)", "{:.2f}"),
]:
    v, t = val_metrics[key], test_metrics[key]
    print(f"{label:<26} {fmt.format(v):>14} {fmt.format(t):>14} "
          f"{fmt.format(t - v):>14}")
print("-" * 84)
print(f"{'Tokens scored':<26} {val_metrics['tokens_scored']:>14,} "
      f"{test_metrics['tokens_scored']:>14,}")
print(f"{'Throughput (tok/s)':<26} {'':>14} "
      f"{test_metrics['tokens_per_second']:>14,.0f}")
print("=" * 84)

# The overfit check: a large test-vs-validation gap means the architecture was
# selected too tightly to the validation set.
gap = test_metrics["loss"] - val_metrics["loss"]
print(f"\nTEST vs VALIDATION GAP: {gap:+.4f}")
if abs(gap) < 0.10:
    print("  Small gap — the validation split was a reliable proxy, so the")
    print("  Stage 6 selection generalised rather than fitting validation noise.")
elif gap > 0:
    print(f"  Test loss is {gap:.4f} HIGHER than validation. The Stage 6")
    print("  selection was partly fitted to validation-set noise. The test")
    print("  figure is the honest one and is what gets reported.")
else:
    print(f"  Test loss is {abs(gap):.4f} LOWER than validation, which happens")
    print("  by chance on small splits. Not evidence of anything.")

---

## Step 5 — Per-group breakdown: where is the model weak?

An aggregate mean can hide a model that performs acceptably on the two largest
domains and unusably on everything else. Since the runtime serves questions
across *all* domains, the weakest group matters as much as the average.

In [ ]:
breakdown = pd.DataFrame([
    {**test_dataset.meta[index], "loss": loss}
    for index, loss in test_metrics["per_record_loss"].items()
])
breakdown["perplexity"] = np.exp(breakdown["loss"].clip(upper=20))

by_domain = (breakdown.groupby("domain")
             .agg(n=("loss", "size"), mean_loss=("loss", "mean"),
                  std_loss=("loss", "std"), mean_ppl=("perplexity", "mean"))
             .sort_values("mean_loss", ascending=False).round(4))

DIFFICULTY_ORDER = ["Beginner", "Intermediate", "Advanced"]
by_difficulty = (breakdown.groupby("difficulty")
                 .agg(n=("loss", "size"), mean_loss=("loss", "mean"),
                      std_loss=("loss", "std"), mean_ppl=("perplexity", "mean"))
                 .reindex([d for d in DIFFICULTY_ORDER
                           if d in breakdown["difficulty"].unique()])
                 .round(4))

print("TEST LOSS BY DOMAIN (worst first)")
print("=" * 78)
print(by_domain.to_string())
print("=" * 78)
print("\nTEST LOSS BY DIFFICULTY")
print("=" * 78)
print(by_difficulty.to_string())
print("=" * 78)

overall = test_metrics["loss"]
worst_domain = by_domain.index[0]
worst_loss = by_domain["mean_loss"].iloc[0]
best_domain = by_domain.index[-1]
best_loss = by_domain["mean_loss"].iloc[-1]

print(f"\n  Overall test loss  : {overall:.4f}")
print(f"  Worst domain       : {worst_domain} ({worst_loss:.4f}, "
      f"{worst_loss / overall:.2f}x the mean)")
print(f"  Best domain        : {best_domain} ({best_loss:.4f})")
print(f"  Spread across domains: {worst_loss - best_loss:.4f}")

if worst_loss > overall * 1.5:
    print(f"\n  WARNING: {worst_domain} is more than 1.5x the mean loss. The")
    print("  aggregate figure overstates performance for this domain, and the")
    print("  runtime should prefer retrieval over generation there.")

In [ ]:
# ── Figure 8.1 — per-group performance ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8),
                         gridspec_kw={"width_ratios": [1.35, 1, 1]})

colours = [PALETTE[3] if v > overall * 1.25 else
           PALETTE[1] if v > overall else PALETTE[2]
           for v in by_domain["mean_loss"]]
bars = axes[0].barh(by_domain.index[::-1], by_domain["mean_loss"][::-1],
                    color=colours[::-1],
                    xerr=by_domain["std_loss"][::-1].fillna(0),
                    error_kw=dict(ecolor="#888888", capsize=3, lw=1.1))
axes[0].axvline(overall, color=PALETTE[0], linestyle="--", linewidth=2,
                label=f"overall = {overall:.3f}")
axes[0].set_title("Test loss by domain (error bars = 1 s.d.)")
axes[0].set_xlabel("Mean cross-entropy loss")
axes[0].legend(fontsize=8)
axes[0].tick_params(axis="y", labelsize=8)
for i, n in enumerate(by_domain["n"][::-1]):
    axes[0].annotate(f"n={n}", xy=(0.02, i), fontsize=7, va="center",
                     color="white", fontweight="bold")

bars = axes[1].bar(by_difficulty.index, by_difficulty["mean_loss"],
                   color=PALETTE[:len(by_difficulty)],
                   yerr=by_difficulty["std_loss"].fillna(0),
                   error_kw=dict(ecolor="#888888", capsize=4, lw=1.1))
axes[1].axhline(overall, color=PALETTE[0], linestyle="--", linewidth=2,
                label=f"overall = {overall:.3f}")
axes[1].set_title("Test loss by difficulty")
axes[1].set_ylabel("Mean cross-entropy loss")
axes[1].bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
axes[1].legend(fontsize=8)
axes[1].margins(y=0.18)

# The distribution matters: a long tail means some questions are far harder.
axes[2].hist(breakdown["loss"], bins=32, color=PALETTE[0],
             edgecolor="white", linewidth=0.5)
axes[2].axvline(overall, color=PALETTE[3], linestyle="--", linewidth=2,
                label=f"mean = {overall:.3f}")
axes[2].axvline(breakdown["loss"].median(), color=PALETTE[2], linestyle=":",
                linewidth=2, label=f"median = {breakdown['loss'].median():.3f}")
axes[2].set_title("Per-question test loss distribution")
axes[2].set_xlabel("Cross-entropy loss for one question")
axes[2].set_ylabel("Number of test questions")
axes[2].legend(fontsize=8)

fig.suptitle("Held-out test performance, broken down by group", y=1.03,
             fontsize=14, fontweight="bold")
fig.tight_layout()
save_figure(fig, "test_breakdown",
            "Red bars mark domains more than 25% above the mean loss. The "
            "aggregate figure alone would conceal them.")
plt.show()

In [ ]:
# ── Figure 8.2 — validation vs test, and the hardest questions ─────────────
fig, axes = plt.subplots(1, 2, figsize=(14.5, 4.8))

metrics_to_plot = ["loss", "top1_accuracy", "top5_accuracy"]
x = np.arange(len(metrics_to_plot))
width = 0.36
val_values = [val_metrics[m] for m in metrics_to_plot]
test_values = [test_metrics[m] for m in metrics_to_plot]

b1 = axes[0].bar(x - width / 2, val_values, width, label="validation",
                 color=PALETTE[1])
b2 = axes[0].bar(x + width / 2, test_values, width, label="test (held out)",
                 color=PALETTE[0])
axes[0].set_xticks(x)
axes[0].set_xticklabels(["Loss\n(lower better)", "Top-1 acc %\n(higher better)",
                         "Top-5 acc %\n(higher better)"], fontsize=8.5)
axes[0].set_title("Validation vs held-out test")
axes[0].legend(fontsize=9)
axes[0].bar_label(b1, fmt="%.2f", padding=2, fontsize=8)
axes[0].bar_label(b2, fmt="%.2f", padding=2, fontsize=8)
axes[0].margins(y=0.18)

hardest = breakdown.nlargest(8, "loss")
bars = axes[1].barh(range(len(hardest)), hardest["loss"][::-1],
                    color=PALETTE[3])
axes[1].set_yticks(range(len(hardest)))
axes[1].set_yticklabels(
    [q[:44] + ("..." if len(q) > 44 else "")
     for q in hardest["question"][::-1]], fontsize=7.5)
axes[1].axvline(overall, color=PALETTE[0], linestyle="--", linewidth=1.8,
                label=f"mean = {overall:.3f}")
axes[1].set_title("The eight hardest test questions")
axes[1].set_xlabel("Cross-entropy loss")
axes[1].legend(fontsize=8)

fig.suptitle("Where the model struggles", y=1.03, fontsize=14,
             fontweight="bold")
fig.tight_layout()
save_figure(fig, "test_vs_val",
            "Right panel names the specific questions the model handled worst, "
            "which is more actionable than an aggregate figure.")
plt.show()

---

## Step 6 — Generation quality on unseen prompts

Loss is a proxy for usefulness, not a measure of it. This step generates from the
model and scores the output on four properties a usable question must have.

The result is reported whatever it is. A model that improves its loss while
producing unusable text has not solved the problem.

In [ ]:
import re

def generate(domain: str, difficulty: str, max_new_tokens: int = 30) -> str:
    prompt = (f"<DOMAIN: {domain}> <DIFFICULTY: {difficulty}>"
              if CONDITIONED else "What")
    ids = tokenizer.encode(prompt, add_special_tokens=True)
    tensor = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        output = model.generate(tensor, max_new_tokens=max_new_tokens,
                                temperature=0.8, top_k=40)
    text = tokenizer.decode(output[0].tolist(), skip_special_tokens=True)
    for token in ("<DOMAIN:", "<DIFFICULTY:", ">", domain, difficulty,
                  "DOMAIN", "DIFFICULTY"):
        text = text.replace(token, " ")
    return " ".join(text.split()).strip()

# Prompts drawn from the test split's own label combinations.
test_prompts = (breakdown[["domain", "difficulty"]]
                .drop_duplicates().head(8).to_dict(orient="records"))

generation_rows = []
print("GENERATION ON UNSEEN TEST LABEL COMBINATIONS")
print("=" * 88)
for prompt in test_prompts:
    text = generate(prompt["domain"], prompt["difficulty"])
    words = text.split()
    distinct_ratio = (len(set(w.lower() for w in words)) / max(len(words), 1))
    generation_rows.append({
        "domain": prompt["domain"],
        "difficulty": prompt["difficulty"],
        "output": text,
        "words": len(words),
        "distinct_ratio": round(distinct_ratio, 3),
        # Four properties a usable interview question must have.
        "has_content": len(words) >= 4,
        "question_shaped": bool(
            text.rstrip().endswith("?") or
            re.match(r"^(what|why|how|when|which|explain|describe|compare)\b",
                     text.strip(), re.IGNORECASE)),
        "not_repetitive": distinct_ratio >= 0.55,
        "readable_length": 4 <= len(words) <= 40,
    })
    print(f"\n  [{prompt['domain']} / {prompt['difficulty']}]")
    print(f"    {text[:140] if text else '(empty output)'}")
print("\n" + "=" * 88)

generation_df = pd.DataFrame(generation_rows)
quality_columns = ["has_content", "question_shaped", "not_repetitive",
                   "readable_length"]
generation_df["passes_all"] = generation_df[quality_columns].all(axis=1)

print("\nGENERATION QUALITY SCORECARD")
print("=" * 88)
print(generation_df[["domain", "difficulty", "words", "distinct_ratio"]
                    + quality_columns + ["passes_all"]].to_string(index=False))
print("=" * 88)

GENERATION_PASS_RATE = float(generation_df["passes_all"].mean())
print(f"\n  Outputs passing every quality check: "
      f"{int(generation_df['passes_all'].sum())} / {len(generation_df)} "
      f"({GENERATION_PASS_RATE * 100:.1f}%)")
for column in quality_columns:
    print(f"    {column:18s} {generation_df[column].mean() * 100:5.1f}% pass")

---

## Step 7 — The promotion gate

Five criteria, with thresholds set before the test split was opened. **All five
must pass.** A gate that can be argued past is not a gate.

Criterion 5 deserves explanation: it requires only that generation is *not
catastrophically broken*, not that it is good. The runtime's design accounts for
weak generation by falling back to retrieval, so a low but non-zero pass rate is
acceptable — but a model producing nothing usable at all must not be promoted.

In [ ]:
GATE_CRITERIA = [
    {
        "name": "test_loss_finite",
        "description": "Test loss is finite and below 10.0",
        "value": test_metrics["loss"],
        "threshold": 10.0,
        "passed": bool(np.isfinite(test_metrics["loss"])
                       and test_metrics["loss"] < 10.0),
        "rationale": "A non-finite or extreme loss means training diverged.",
    },
    {
        "name": "beats_uniform_baseline",
        "description": "Test perplexity is below the uniform-guess baseline",
        "value": test_metrics["perplexity"],
        "threshold": float(getattr(tokenizer, "vocab_size", 4096)),
        "passed": bool(test_metrics["perplexity"]
                       < float(getattr(tokenizer, "vocab_size", 4096))),
        "rationale": ("A model that cannot beat guessing uniformly over the "
                      "vocabulary has learned nothing at all."),
    },
    {
        "name": "top5_accuracy_minimum",
        "description": "Top-5 next-token accuracy is at least 15%",
        "value": test_metrics["top5_accuracy"],
        "threshold": 15.0,
        "passed": bool(test_metrics["top5_accuracy"] >= 15.0),
        "rationale": ("The correct token must be in the model's top five at a "
                      "usable rate for generation to function."),
    },
    {
        "name": "no_severe_val_test_gap",
        "description": "Test loss exceeds validation loss by less than 1.0",
        "value": abs(gap),
        "threshold": 1.0,
        "passed": bool(abs(gap) < 1.0),
        "rationale": ("A large gap means the Stage 6 selection was fitted to "
                      "validation noise, so the test result would not "
                      "generalise."),
    },
    {
        "name": "generation_not_broken",
        "description": "At least one generated output passes every quality check",
        "value": GENERATION_PASS_RATE * 100,
        "threshold": 1.0,
        "passed": bool(GENERATION_PASS_RATE > 0.0),
        "rationale": ("Weak generation is acceptable because the runtime falls "
                      "back to retrieval, but a model producing nothing usable "
                      "must not be promoted."),
    },
]

print("=" * 92)
print("PROMOTION GATE")
print("=" * 92)
for i, criterion in enumerate(GATE_CRITERIA, 1):
    status = "PASS" if criterion["passed"] else "FAIL"
    print(f"\n  {i}. [{status}] {criterion['description']}")
    print(f"          measured {criterion['value']:.4f}  "
          f"threshold {criterion['threshold']:.4f}")
    print(f"          why: {criterion['rationale']}")

PROMOTION_APPROVED = all(c["passed"] for c in GATE_CRITERIA)
failed = [c["name"] for c in GATE_CRITERIA if not c["passed"]]

print("\n" + "=" * 92)
if PROMOTION_APPROVED:
    print(f"  PROMOTION APPROVED — all {len(GATE_CRITERIA)} criteria passed.")
    print(f"  The {MODEL_KIND} model may be registered in Stage 9.")
else:
    print(f"  PROMOTION DENIED — {len(failed)} criterion/criteria failed:")
    for name in failed:
        print(f"      - {name}")
    print("\n  Stage 9 will NOT register this model. The runtime continues to")
    print("  serve questions by retrieval over the labelled dataset, which is")
    print("  the designed fallback rather than an outage.")
print("=" * 92)

In [ ]:
# ── Figure 8.3 — the gate ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8),
                         gridspec_kw={"width_ratios": [1.1, 1]})

names = [c["name"].replace("_", "\n") for c in GATE_CRITERIA]
# Normalise each criterion to "fraction of its threshold met", so five
# criteria on different scales can be shown on one axis.
ratios = []
for c in GATE_CRITERIA:
    if c["name"] in ("test_loss_finite", "no_severe_val_test_gap",
                     "beats_uniform_baseline"):
        # Lower is better: show headroom below the threshold.
        ratio = min(2.0, c["threshold"] / max(c["value"], 1e-9))
    else:
        ratio = min(2.0, c["value"] / max(c["threshold"], 1e-9))
    ratios.append(ratio)

colours = [PALETTE[2] if c["passed"] else PALETTE[3] for c in GATE_CRITERIA]
bars = axes[0].barh(names[::-1], ratios[::-1], color=colours[::-1])
axes[0].axvline(1.0, color="black", linestyle="--", linewidth=2,
                label="threshold (1.0 = exactly met)")
axes[0].set_title("Promotion criteria — margin against threshold")
axes[0].set_xlabel("Ratio to threshold (>1 = passed with headroom)")
axes[0].tick_params(axis="y", labelsize=7.5)
axes[0].legend(fontsize=8)
axes[0].bar_label(bars, fmt="%.2fx", padding=3, fontsize=8)
axes[0].margins(x=0.18)

axes[1].axis("off")
verdict = "APPROVED" if PROMOTION_APPROVED else "DENIED"
verdict_colour = PALETTE[2] if PROMOTION_APPROVED else PALETTE[3]
axes[1].text(0.5, 0.78, verdict, ha="center", va="center", fontsize=34,
             fontweight="bold", color=verdict_colour,
             transform=axes[1].transAxes)
axes[1].text(0.5, 0.60,
             f"{sum(c['passed'] for c in GATE_CRITERIA)} of "
             f"{len(GATE_CRITERIA)} criteria passed",
             ha="center", va="center", fontsize=12,
             transform=axes[1].transAxes)
summary_lines = [
    f"Model        : {MODEL_KIND}",
    f"Architecture : {selection['selected']['label']}",
    f"Test loss    : {test_metrics['loss']:.4f}",
    f"Perplexity   : {test_metrics['perplexity']:.2f}",
    f"Top-1 acc    : {test_metrics['top1_accuracy']:.2f}%",
    f"Top-5 acc    : {test_metrics['top5_accuracy']:.2f}%",
    f"Generation   : {GENERATION_PASS_RATE * 100:.0f}% fully usable",
]
axes[1].text(0.5, 0.26, "\n".join(summary_lines), ha="center", va="center",
             fontsize=10, family="monospace", transform=axes[1].transAxes,
             bbox=dict(boxstyle="round,pad=0.7", fc="#F5F7FA", ec="#CCCCCC"))

fig.suptitle("Held-out evaluation verdict", y=1.03, fontsize=14,
             fontweight="bold")
fig.tight_layout()
save_figure(fig, "promotion_gate",
            "All five criteria must clear 1.0x for promotion.")
plt.show()

---

## Step 8 — Evaluation report

In [ ]:
evaluation = {
    "stage": "08_held_out_test_evaluation",
    "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "test_split_access": {
        "authorised_notebook": NOTEBOOK_ID,
        "seal_sha256": seal["sha256"],
        "integrity_verified": True,
        "sealed_utc": seal["sealed_utc"],
        "note": "First and only authorised read of the test split.",
    },
    "model": {
        "kind": MODEL_KIND,
        "architecture": selection["selected"]["label"],
        "candidate_id": selection["selected"]["candidate_id"],
        "checkpoint": str(CKPT_PATH.relative_to(WORKSPACE_DIR)),
        "parameters": int(model.count_parameters()),
        "input_format": ("conditioned" if CONDITIONED else "plain"),
    },
    "test_metrics": {
        k: (round(v, 5) if isinstance(v, float) else v)
        for k, v in test_metrics.items() if k != "per_record_loss"
    },
    "validation_metrics": {
        k: (round(v, 5) if isinstance(v, float) else v)
        for k, v in val_metrics.items() if k != "per_record_loss"
    },
    "val_test_gap": round(gap, 5),
    "by_domain": by_domain.reset_index().to_dict(orient="records"),
    "by_difficulty": by_difficulty.reset_index().to_dict(orient="records"),
    "weakest_domain": {"domain": worst_domain, "mean_loss": float(worst_loss)},
    "generation": {
        "samples": generation_df.to_dict(orient="records"),
        "pass_rate": round(GENERATION_PASS_RATE, 4),
        "criteria": quality_columns,
    },
    "promotion_gate": {
        "criteria": [
            {k: (float(v) if isinstance(v, (int, float, np.floating))
                 and k in ("value", "threshold") else v)
             for k, v in c.items()}
            for c in GATE_CRITERIA
        ],
        "approved": bool(PROMOTION_APPROVED),
        "failed_criteria": failed,
    },
    "limitations": [
        "The training corpus is small for from-scratch language modelling, so "
        "absolute perplexity is not comparable to models trained on billions "
        "of tokens.",
        "Generation fluency is limited; the runtime mitigates this with a "
        "quality gate and retrieval fallback.",
        f"The test split contains {len(test_records)} records, so per-domain "
        f"figures rest on small samples and carry wide uncertainty.",
    ],
}

report_path = REPORTS_DIR / "fine_tuned_model_evaluation.json"
report_path.write_text(json.dumps(evaluation, indent=2, default=str),
                       encoding="utf-8")

print(f"Evaluation report : {report_path.relative_to(WORKSPACE_DIR)}")
print(f"Promotion         : {'APPROVED' if PROMOTION_APPROVED else 'DENIED'}")
print(f"Figures           : "
      f"{len(sorted(FIGURES_DIR.glob(f'{NOTEBOOK_ID:02d}_*.png')))}")

---

## Stage 8 summary

| Aspect | Result |
|---|---|
| Test-split access | authorised (Notebook 08 only), seal integrity verified |
| Metrics | loss, perplexity, top-1 and top-5 accuracy |
| Group breakdown | per domain and per difficulty, weakest group named |
| Generation check | four usability criteria, pass rate reported |
| Promotion gate | five criteria, all must pass |

### Why this evaluation is trustworthy

1. **The test split was read once**, by the only notebook authorised to do so,
   and its hash was verified against the Stage 4 seal.
2. **The model was chosen before the test data was opened** — Stage 6 selected
   on validation, Stage 7 nominated on validation. Nothing here re-chose.
3. **The gate thresholds were declared before the read**, so they could not be
   adjusted to fit the result.
4. **The weakest group is named**, not averaged away.

### Limitations, stated

The corpus is small for from-scratch language modelling, the test split is
small enough that per-domain figures carry wide uncertainty, and generation
fluency is limited. All three are recorded in the evaluation report rather than
left for a reader to discover.

### Next

**Notebook 09 — Model Export and Registration**, which packages the model only
if this gate approved it.